# 00g Promotion And Release Gate

Purpose:
- verify acceptance gate + production readiness from service endpoints,
- create/find model-registry entry for the selected model,
- promote champion only if gate passes,
- export a final release evidence snapshot for audit.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import requests

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS')
REPORTS_DIR = ROOT / 'Ai miroservices' / 'modeling' / 'outputs' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_BASE = 'http://localhost:8091'
DATASET = 'B'
MODEL_NAME = 'CATBOOST'
MODEL_VERSION = 'v1'
SPLIT = 'test'
INFERENCE_WINDOW = 200
SOAK_HOURS = 24

session = requests.Session()
session.timeout = 30


In [2]:
def get_json(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def post_json(url, body):
    r = session.post(url, json=body, timeout=30)
    r.raise_for_status()
    return r.json()


In [3]:
acceptance = get_json(
    f'{FORECAST_BASE}/artifacts/acceptance-gate',
    params={
        'dataset': DATASET,
        'model_name': MODEL_NAME,
        'split': SPLIT,
        'inference_window': INFERENCE_WINDOW,
    },
)
readiness = get_json(
    f'{FORECAST_BASE}/artifacts/production-readiness',
    params={
        'dataset': DATASET,
        'model_name': MODEL_NAME,
        'split': SPLIT,
        'inference_window': INFERENCE_WINDOW,
        'soak_hours': SOAK_HOURS,
    },
)

print('acceptance_ready:', acceptance.get('ready'))
print('production_ready:', readiness.get('ready'))


In [4]:
registry = get_json(f'{FORECAST_BASE}/model-registry', params={'dataset': DATASET})
items = registry.get('items', [])

entry = None
for it in items:
    if str(it.get('model_name', '')).upper() == MODEL_NAME and str(it.get('dataset')) == DATASET:
        entry = it
        break

if entry is None:
    created = post_json(
        f'{FORECAST_BASE}/model-registry',
        {
            'dataset': DATASET,
            'warehouse_id': None,
            'model_name': MODEL_NAME,
            'model_version': MODEL_VERSION,
            'artifact_stage': 'production',
            'status': 'active',
            'is_champion': False,
            'priority': 1,
        },
    )
    entry_id = int(created['id'])
    print('created registry entry:', entry_id)
else:
    entry_id = int(entry['id'])
    print('existing registry entry:', entry_id)

entry_id

existing registry entry: 1


1

In [5]:
promotion_result = {'attempted': False, 'skipped_reason': None, 'response': None}
if bool(readiness.get('ready')):
    promotion_result['attempted'] = True
    promotion_result['response'] = post_json(
        f'{FORECAST_BASE}/model-registry/promote',
        {
            'entry_id': entry_id,
            'split': SPLIT,
            'inference_window': INFERENCE_WINDOW,
        },
    )
else:
    promotion_result['skipped_reason'] = 'production_readiness_not_ready'

promotion_result

{'attempted': True,
 'skipped_reason': None,
 'response': {'ok': True,
  'entry_id': 1,
  'dataset': 'B',
  'warehouse_id': None,
  'model_name': 'CATBOOST',
  'model_version': 'v1'}}

In [6]:
release_evidence = get_json(
    f'{FORECAST_BASE}/artifacts/release-evidence',
    params={
        'dataset': DATASET,
        'model_name': MODEL_NAME,
        'split': SPLIT,
        'inference_window': INFERENCE_WINDOW,
        'soak_hours': SOAK_HOURS,
        'history_limit': 50,
    },
)

snapshot = {
    'timestamp_utc': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'dataset': DATASET,
    'model_name': MODEL_NAME,
    'split': SPLIT,
    'inference_window': INFERENCE_WINDOW,
    'soak_hours': SOAK_HOURS,
    'acceptance_gate': acceptance,
    'production_readiness': readiness,
    'promotion_result': promotion_result,
    'release_evidence': release_evidence,
}

out = REPORTS_DIR / 'promotion_and_release_gate_snapshot.json'
out.write_text(json.dumps(snapshot, indent=2), encoding='utf-8')
print('WROTE', out)
print('overall_ok:', release_evidence.get('summary', {}).get('overall_ok'))
